# 02 - Train and evaluate YOLOv9-C

This notebook runs the fixed project workflow: YOLOv9-C, 640px images, batch 4, 10 epochs, seed 42, and confidence 0.25 for visual predictions. A pretrained checkpoint supplies transfer-learning weights; the YAML supplies the architecture; `train_dual.py` runs training.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path
from IPython.display import Image as DisplayImage, display

NOTEBOOK_DIR = Path.cwd()
PROJECT = NOTEBOOK_DIR.parent
YOLO_DIR = PROJECT / 'yolov9'
DATA_YAML = PROJECT / 'roadwatcher.yaml'
DATA_DIR = PROJECT / 'data_clean'
IMG_SIZE, BATCH_SIZE, EPOCHS, SEED = 640, 4, 10, 42
RUN_NAME = 'roadwatcher_yolov9c'
assert DATA_YAML.exists(), 'Run notebooks/01_prepare_data.ipynb first.'
os.chdir(YOLO_DIR)
WEIGHTS = YOLO_DIR / 'yolov9-c.pt'
print('Working directory:', Path.cwd())

## Environment

CUDA is required for this project. The install command uses the notebook kernel's Python executable.

In [ ]:
!nvidia-smi
!{sys.executable} -m pip install -q -r requirements.txt
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Select the Python kernel on the Vast.ai GPU machine.')
print(torch.cuda.get_device_name(0))
if not WEIGHTS.exists():
    urllib.request.urlretrieve('https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt', WEIGHTS)
print('Weights:', WEIGHTS)

## Train

Ten epochs means ten passes over the training set. Batch size 4 controls how many images contribute to each update. This is fine-tuning because training starts from learned COCO weights.

In [ ]:
!{sys.executable} train_dual.py --device 0 --workers 2 --img {IMG_SIZE} --batch-size {BATCH_SIZE} --epochs {EPOCHS} --data {DATA_YAML} --cfg models/detect/yolov9-c.yaml --weights {WEIGHTS} --hyp data/hyps/hyp.scratch-high.yaml --name {RUN_NAME} --min-items 0 --close-mosaic 2 --exist-ok
BEST = YOLO_DIR / 'runs' / 'train' / RUN_NAME / 'weights' / 'best.pt'
LAST = YOLO_DIR / 'runs' / 'train' / RUN_NAME / 'weights' / 'last.pt'
assert BEST.exists(), BEST
assert LAST.exists(), LAST
display(DisplayImage(filename=str(YOLO_DIR / 'runs' / 'train' / RUN_NAME / 'results.png')))

## Evaluate on test

Precision is the fraction of predicted objects that are correct. Recall is the fraction of labelled objects found. mAP@50 uses IoU 0.50; mAP@50-95 averages IoU thresholds from 0.50 to 0.95 and is stricter. Evaluation uses `0.001` to measure the confidence ranking broadly; visualisation below uses the required `0.25` threshold.

In [ ]:
!{sys.executable} val_dual.py --data {DATA_YAML} --weights {BEST} --img 640 --batch-size 4 --conf-thres 0.001 --iou-thres 0.7 --task test --verbose --exist-ok

## Visual predictions

In [ ]:
PREDICTION_DIR = YOLO_DIR / 'runs' / 'detect' / 'roadwatcher_test'
!{sys.executable} detect_dual.py --weights {BEST} --source {DATA_DIR / 'test' / 'images'} --data {DATA_YAML} --img 640 --conf-thres 0.25 --iou-thres 0.45 --project runs/detect --name roadwatcher_test --exist-ok
prediction_images = sorted(p for p in PREDICTION_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})[:6]
for prediction in prediction_images:
    display(DisplayImage(filename=str(prediction), width=480))

## Output locations

| Output | Location |
|---|---|
| Best checkpoint | `yolov9/runs/train/roadwatcher_yolov9c/weights/best.pt` |
| Last checkpoint | `yolov9/runs/train/roadwatcher_yolov9c/weights/last.pt` |
| Training plots | `yolov9/runs/train/roadwatcher_yolov9c/results.png` |
| Predictions | `yolov9/runs/detect/roadwatcher_test/` |

Download these files in VS Code with right-click > Download before destroying the Vast.ai instance. Instance storage is lost when it is destroyed.